# Phase 3 - Unified Model Optimization

This notebook consolidates three distinct hyperparameter optimization strategies explored during development:

1. **Baseline Grid Search** : Manual exploration of key architectural parameters
2. **Bayesian Optimization with Optuna** : Automated search using Optuna's TPE sampler
3. **Extended Search Space** : Comprehensive parameter exploration including layer depth and activation functions

Each section preserves the original experimental intent while using shared utility functions from the `src` module for consistency.

## Environment Setup

Load required libraries and configure the project path for consistent imports.

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import optuna
import joblib
from optuna.trial import TrialState
from sklearn.model_selection import train_test_split

# Project imports
src_dir = Path("../src")
sys.path.insert(0, str(src_dir))

from data_preprocessing import load_data, resolve_project_root
from evaluate import generate_anomaly_metrics_and_threshold
from model import build_autoencoder, Autoencoder
from train import set_global_seed, train_autoencoder

# Initialize project paths
project_root = resolve_project_root()
data_dir = project_root / "data" / "processed"
models_dir = project_root / "models"

# Set global seeds for reproducibility
set_global_seed(69)

# Notebook configuration
RUN_MODE = True  # Set to True to train all models, False to load a pre-trained model
N_TRIALS = 50  # Number of trials for hyperparameter optimization

## Data Loading

Load the preprocessed training and test datasets for optimization experiments.

In [11]:
# Load train and test tables
train_data_path = data_dir / "train.csv"
test_data_path = data_dir / "test.csv"

train_set = load_data(train_data_path)
test_set = load_data(test_data_path)

print("Train Set:")
display(train_set.head())
print("Test Set:")
display(test_set.head())

# Convert labels to numeric classes expected by sklearn metrics
label_to_int = {"normal": 0, "anomaly": 1}

feature_cols = [c for c in train_set.columns if c not in ["label", "binary_label"]]
X_train_df = train_set[feature_cols]
X_test_df = test_set[feature_cols]

y_train_series = train_set["binary_label"].map(label_to_int)
y_test_series = test_set["binary_label"].map(label_to_int)

if y_train_series.isna().any() or y_test_series.isna().any():
    raise ValueError("Unexpected values found in binary_label. Expected only 'normal' and 'anomaly'.")

# Default source: arrays built from CSV
X_train = X_train_df.to_numpy(dtype=np.float32)
X_test = X_test_df.to_numpy(dtype=np.float32)
y_train = y_train_series.to_numpy(dtype=np.int32)
y_test = y_test_series.to_numpy(dtype=np.int32)
source = "CSV-derived arrays"

print(f"Train Set Shape: {X_train.shape}, Labels Shape: {y_train.shape}")
print(f"Test Set Shape: {X_test.shape}, Labels Shape: {y_test.shape}")

Train Set:


,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,-0.145561,-0.026526,-0.092098,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
1,-0.145561,-0.026472,-0.051431,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
2,-0.145561,-0.026689,-0.086207,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
3,-0.145561,-0.026173,-0.066131,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
4,-0.145561,-0.025901,0.015446,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False


Test Set:


,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,-0.145561,-0.035007,-0.098556,0.0,0.0,0.0,-0.053347,-0.010067,-1.959581,-0.008538,...,False,False,False,False,True,False,False,False,False,False
1,-0.145561,-0.035007,-0.098556,0.0,0.0,0.0,-0.053347,-0.010067,-1.959581,-0.008538,...,False,False,False,False,True,False,False,False,False,False
2,-0.145561,-0.035007,-0.098556,0.0,0.0,0.0,-0.053347,-0.010067,-1.959581,-0.008538,...,True,False,False,False,False,False,False,False,False,False
3,-0.145561,-0.029163,-0.080533,0.0,0.0,0.0,-0.053347,-0.010067,0.510313,-0.008538,...,False,False,False,False,False,False,False,False,True,False
4,-0.145561,-0.035007,-0.098556,0.0,0.0,0.0,-0.053347,-0.010067,-1.959581,-0.008538,...,False,False,False,False,True,False,False,False,False,False


Train Set Shape: (61482, 118), Labels Shape: (61482,)
Test Set Shape: (84104, 118), Labels Shape: (84104,)


## Section 1: Grid Search Optimization (v1)

This section implements the original manual grid search approach, exploring a predefined set of hyperparameters to establish baseline optmization performance bounds.

**Search Strategy:**
- Latent dimension: [8, 16, 32]
- Learning rate: [1e-3, 5e-4, 1e-4]
- Batch size: [128, 256, 512]

**Approach:** Train and evaluate each combination manually, tracking the best validation loss.

In [13]:
if RUN_MODE:
    # Define search space for baseline grid search
    latent_dims = [8, 16, 32]
    learning_rates = [1e-3, 5e-4, 1e-4]
    batch_sizes = [128, 256, 512]

    # Storage for results
    v1_results = []
    best_val_loss = float('inf')
    best_params = None
    best_model = None

    # Grid search execution
    total_combinations = len(latent_dims) * len(learning_rates) * len(batch_sizes)
    print(f"Testing {total_combinations} hyperparameter combinations...")

    combo_count = 0
    for latent_dim in latent_dims:
        for lr in learning_rates:
            for batch_size in batch_sizes:
                combo_count += 1
                print(f"\nCombination {combo_count}/{total_combinations}:")
                print(f"  Latent dim: {latent_dim}, LR: {lr}, Batch size: {batch_size}")

                # Split training data for validation
                x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

                # Build model
                model = build_autoencoder(
                    input_dim=X_train.shape[1],
                    latent_dim=latent_dim,
                    hidden_units=32,
                    dropout_rate=0.0,
                    n_hidden_layers=1,
                    activation_encoder='relu',
                    activation_decoder='relu'
                )

                # Compile model
                optimizer = keras.optimizers.Adam(learning_rate=lr)
                model.compile(optimizer=optimizer, loss='mse')

                # Train model
                history = model.fit(
                    x_fit, x_fit,
                    validation_data=(x_val, x_val),
                    epochs=20,
                    batch_size=batch_size,
                    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
                    verbose=0
                )

                # Record results
                val_loss = min(history.history['val_loss'])
                v1_results.append({
                    'latent_dim': latent_dim,
                    'learning_rate': lr,
                    'batch_size': batch_size,
                    'val_loss': val_loss,
                    'epochs': len(history.history['loss'])
                })

                # Track best model
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_params = {'latent_dim': latent_dim, 'learning_rate': lr, 'batch_size': batch_size}
                    best_model = model
                    print(f"  -> New best val_loss: {val_loss:.6f}")
                else:
                    print(f"  -> Val loss: {val_loss:.6f}")

    print("\n" + "="*50)
    print("BASELINE GRID SEARCH COMPLETE")
    print("="*50)
    print(f"Best validation loss: {best_val_loss:.6f}")
    print(f"Best parameters: {best_params}")
    
    # Convert results to DataFrame for analysis
    v1_df = pd.DataFrame(v1_results)
    print("\nTop 5 configurations:")
    print(v1_df.nsmallest(5, 'val_loss')[['latent_dim', 'learning_rate', 'batch_size', 'val_loss']].to_string(index=False))
    
    
    # Train the best model on the full training set with early stopping
    print("\nPreparing best model for full training...")
    best_history = best_model.fit(
        X_train, X_train,
        validation_split=0.1,
        epochs=100,
        batch_size=best_params['batch_size'],
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0
    )
    
    # Best Model Summary
    print("\nBest Model Summary:")
    best_model.summary()
    
    # Print 
    print("\nBest Model Parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}") 
        
        
    # Save best model parameters to a text file
    print("\nSaving best model parameters to disk...")
    with open(models_dir / "best_model_params_v1.txt", "w") as f:
        f.write("Best Model Parameters:\n")
        for key, value in best_params.items():
            f.write(f"{key}: {value}\n")    
    print("Best model parameters saved successfully.")
    
    
    # Save the best model to disk
    print("\nSaving best model to disk...")
    best_model.save(models_dir / "autoencoder_best_model_v1.keras")
    print("Best model saved successfully.")
    
    
else:
    print("Loading best model data from disk...")
    final_model = keras.models.load_model(models_dir / "autoencoder_best_model_v1.keras")

    # Load best model parameters from the text file
    best_params = {}
    with open(models_dir / "best_model_params_v1.txt", "r") as f:
        lines = f.readlines()[1:]  # Skip the first line
        for line in lines:
            key, value = line.strip().split(": ")
            best_params[key] = float(value) if '.' in value else int(value)
    
    # Print best model parameters
    print("\nBest Model Parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
                  
    print("\nBest Model Summary:")
    final_model.summary()

    
    

Testing 27 hyperparameter combinations...

Combination 1/27:
  Latent dim: 8, LR: 0.001, Batch size: 128
  -> New best val_loss: 0.054233

Combination 2/27:
  Latent dim: 8, LR: 0.001, Batch size: 256
  -> New best val_loss: 0.053436

Combination 3/27:
  Latent dim: 8, LR: 0.001, Batch size: 512
  -> Val loss: 0.059053

Combination 4/27:
  Latent dim: 8, LR: 0.0005, Batch size: 128
  -> Val loss: 0.063214

Combination 5/27:
  Latent dim: 8, LR: 0.0005, Batch size: 256
  -> Val loss: 0.066585

Combination 6/27:
  Latent dim: 8, LR: 0.0005, Batch size: 512
  -> Val loss: 0.087816

Combination 7/27:
  Latent dim: 8, LR: 0.0001, Batch size: 128
  -> Val loss: 0.131291

Combination 8/27:
  Latent dim: 8, LR: 0.0001, Batch size: 256
  -> Val loss: 0.156141

Combination 9/27:
  Latent dim: 8, LR: 0.0001, Batch size: 512
  -> Val loss: 0.191348

Combination 10/27:
  Latent dim: 16, LR: 0.001, Batch size: 128
  -> New best val_loss: 0.019947

Combination 11/27:
  Latent dim: 16, LR: 0.001, Batc

Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 118)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Sequential)            │ (None, 32)             │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Sequential)            │ (None, 118)            │         4,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,444 (115.02 KB)

 Trainable params: 9,814 (38.34 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 19,630 (76.68 KB)


Best Model Parameters:
  latent_dim: 32
  learning_rate: 0.001
  batch_size: 128

Saving best model parameters to disk...
Best model parameters saved successfully.

Saving best model to disk...
Best model saved successfully.


## Section 2: Bayesian Optimization with Optuna

This section implements automated hyperparameter optimization using Optuna's Tree-structured Parzen Estimator (TPE) sampler.

**Search Strategy:**
- Latent dimension: [8, 16, 32, 64]
- Learning rate: log-uniform range [1e-4, 1e-2]
- Batch size: [32, 64, 128, 256, 512]
- Dropout rate: [0.0, 0.1, 0.2, 0.3]
- Number of hidden layers: [1, 2, 3]

**Approach:**
Optuna intelligently, the hyperparameter space, balancing exploration and exploitation to find optimal configurations efficiently.

In [ ]:
if RUN_MODE:
    # Define search space
    search_space_2 = {
        'latent_dim': [8, 16, 32, 64],
        'learning_rate': [1e-2, 1e-4], 
        'batch_size': [32, 64, 128, 256, 512],
        'dropout_rate': [0.0, 0.1, 0.2, 0.3],
        'n_hidden_layers': [1, 2, 3],
        'hidden_units': [16, 32, 64, 128], # Direct specification instead of geometric 
    }

    # Output the search space for verification
    print("\nHyperparameter Search Space:")
    for param, values in search_space_2.items():
        print(f"{param}: {values}")# 
    print("\n" + "="*50)    

    # Prepare validation split for Optuna objective
    x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

    # Define Optuna objective function
    def objective(trial):
        """Objective function for Optuna optimization."""
        # Sample hyperparameters
        latent_dim = trial.suggest_categorical('latent_dim', search_space_2['latent_dim'])
        learning_rate = trial.suggest_loguniform('learning_rate', search_space_2['learning_rate'][1], search_space_2['learning_rate'][0])
        batch_size = trial.suggest_categorical('batch_size', search_space_2['batch_size'])
        dropout_rate = trial.suggest_categorical('dropout_rate', search_space_2['dropout_rate'])
        n_hidden_layers = trial.suggest_categorical('n_hidden_layers', search_space_2['n_hidden_layers'])
        hidden_units = trial.suggest_categorical('hidden_units', search_space_2['hidden_units'])

        # Build model with sampled hyperparameters
        model = build_autoencoder(
            input_dim=X_train.shape[1],
            latent_dim=latent_dim,
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            n_hidden_layers=n_hidden_layers,
            activation_encoder='relu',
            activation_decoder='relu'
        )

        # Compile model
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='mse')

        # Train model with early stopping
        history = model.fit(
            x_fit, x_fit,
            validation_data=(x_val, x_val),
            epochs=30,
            batch_size=batch_size,
            callbacks=[
                keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
            ],
            verbose=0
        )

        # Return validation loss for minimization
        return min(history.history['val_loss'])

    # Create and run Optuna study
    print("Starting Optuna Bayesian optimization...")
    study2 = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study2.optimize(objective, n_trials=N_TRIALS, timeout=800)  # 50 trials or 15 minutes max

    # Display results
    print("\n" + "="*50)
    print("OPTUNA BAYESIAN OPTIMIZATION COMPLETE")
    print("="*50)
    print(f"Number of finished trials: {len(study2.trials)}")

    print("\nBest trial:")
    trial = study2.best_trial
    print(f"  Value (validation loss): {trial.value:.6f}")
    print("  Parameters:")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
        
        
    # Train the best model on the full training set with early stopping
    print("\nPreparing best model for full training...")
        
    # Get best parameters
    best_config = study2.best_params
    best_val_loss = study2.best_value
    
    # Train final model with best config on full training data
    print("\nTraining final model with best hyperparameters...")
    final_model = build_autoencoder(
        input_dim=X_train.shape[1],
        latent_dim=best_config['latent_dim'],
        dropout_rate=best_config['dropout_rate'],
        n_hidden_layers=best_config['n_hidden_layers'],
        hidden_units=best_config['hidden_units'],
        activation_encoder='relu',
        activation_decoder='relu'
    )

    final_model.compile(optimizer=keras.optimizers.Adam(learning_rate=best_config['learning_rate']), loss='mse')

    # Train longer on full training data with validation split
    final_history = final_model.fit(
        X_train, X_train,
        validation_split=0.1,
        epochs=100,
        batch_size=best_config['batch_size'],
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=1
    )

    # Save best model parameters to a text file
    print("\nSaving best model parameters to disk...")
    with open(models_dir / "best_model_params_v2.txt", "w") as f:
        f.write("Best Model Parameters:\n")
        for key, value in best_config.items():
            f.write(f"{key}: {value}\n")
    print("Best model parameters saved successfully.")


    # Save the final model
    final_model.save(models_dir / "autoencoder_best_model_v2.keras")
    print("Final model saved!")
    
    # Save the final study for future reference
    joblib.dump(study2, models_dir / "optuna_study_v2.pkl")
    print("Optuna study saved successfully.")
    
    # Visualize optimization history
    try:
        fig = optuna.visualization.plot_optimization_history(study2)
        fig.show()
    except Exception as e:
        print(f"Could not generate optimization history plot: {e}")

    try:
        fig = optuna.visualization.plot_param_importances(study2)
        fig.show()
    except Exception as e:
        print(f"Could not generate parameter importance plot: {e}")
else:
    # Load best model and parameters from disk
    print("Loading best model data from disk...")
    final_model = keras.models.load_model(models_dir / "autoencoder_best_model_v2.keras")

    # Load the Optuna study
    study2 = joblib.load(models_dir / "optuna_study_v2.pkl")
    print("Optuna study loaded successfully.")

    # Load best model parameters from the text file
    best_params = {}
    with open(models_dir / "best_model_params_v2.txt", "r") as f:
        lines = f.readlines()[1:]  # Skip the first line
        for line in lines:
            key, value = line.strip().split(": ")
            best_params[key] = float(value) if '.' in value else int(value)
    
    # Print best model parameters
    print("\nBest Model Parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
                
    print("\nBest Model Summary:")
    final_model.summary()

[I 2026-07-29 23:58:38,642] A new study created in memory with name: no-name-499f29ba-ee30-462f-b8de-24f6f4b8a492



Hyperparameter Search Space:
latent_dim: [8, 16, 32, 64]
learning_rate: [0.01, 0.0001]
batch_size: [32, 64, 128, 256, 512]
dropout_rate: [0.0, 0.1, 0.2, 0.3]
n_hidden_layers: [1, 2, 3]
hidden_units: [16, 32, 64, 128]

Starting Optuna Bayesian optimization...


/var/folders/cp/98185ksd5djf61ly4251cnw40000gn/T/ipykernel_28343/465175023.py:26: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', search_space_2['learning_rate'][1], search_space_2['learning_rate'][0])
[I 2026-07-29 23:58:58,671] Trial 0 finished with value: 0.023951668292284012 and parameters: {'latent_dim': 16, 'learning_rate': 0.0002051338263087451, 'batch_size': 128, 'dropout_rate': 0.1, 'n_hidden_layers': 3, 'hidden_units': 128}. Best is trial 0 with value: 0.023951668292284012.
[I 2026-07-29 23:59:01,253] Trial 1 finished with value: 0.14484471082687378 and parameters: {'latent_dim': 64, 'learning_rate': 0.0037183641805732083, 'batch_size': 512, 'dropout_rate': 0.3, 'n_hidden_layers': 1, 'hidden_units': 16}. Best is trial 0 with value: 0.023951668292284012.
[I 2026-0


OPTUNA BAYESIAN OPTIMIZATION COMPLETE
Number of finished trials: 50

Best trial:
  Value (validation loss): 0.005644
  Parameters:
    latent_dim: 64
    learning_rate: 0.0006678810092336423
    batch_size: 512
    dropout_rate: 0.0
    n_hidden_layers: 1
    hidden_units: 64

Preparing best model for full training...

Training final model with best hyperparameters...
Epoch 1/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2457 - val_loss: 0.1472
Epoch 2/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1261 - val_loss: 0.0791
Epoch 3/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0740 - val_loss: 0.0555
Epoch 4/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0487 - val_loss: 0.0409
Epoch 5/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0354 - val_loss: 0.0304
Epoch 6/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0268 - val_loss: 0.0233
Epoch 7/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0210 - val_loss: 0.0183
Epoch 8/100
109/109

## Section 3: Extended Search Space Exploration (v3)

This section explores an expanded hyperparameter space including architectural variations and different activation functions to discover potentially superior model configurations.

**Extended Search Space:**
- Latent dimension: [8, 16, 32, 64, 128]
- Learning rate: [1e-2, 5e-3, 1e-3, 5e-4, 1e-4]
- Batch size: [32, 64, 128, 256, 512, 1024]
- Dropout rate: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
- Hidden layers: [1, 2, 3, 4]
- Hidden units: [16, 32, 64, 128, 256]
- Encoder activation: ['relu', 'tanh', 'sigmoid', 'leaky_relu']
- Decoder activation: ['relu', 'sigmoid', 'linear', 'tanh', 'leaky_relu']

**Approach:**
Systematic exploration of the extended space using random sampling to identify promising regions for further investigation.

In [19]:
if RUN_MODE:
    # Define extended search space
    # search_space_3 = {
    #     'latent_dim': [8, 16, 32, 64, 128],
    #     'learning_rate': [1e-2, 5e-3, 1e-3, 5e-4, 1e-4],
    #     'batch_size': [32, 64, 128, 256, 512, 1024],
    #     'dropout_rate': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    #     'n_hidden_layers': [1, 2, 3, 4],
    #     'hidden_units': [16, 32, 64, 128, 256],
    #     'activation_encoder': ['relu', 'tanh', 'sigmoid','leaky_relu'],
    #     'activation_decoder': ['relu', 'sigmoid', 'linear', 'tanh','leaky_relu']
    # }

    search_space_3 = {
    'latent_dim': [16, 32, 64],
    'learning_rate': [1e-3, 5e-4, 1e-4],
    'batch_size': [128, 256, 512],
    'dropout_rate': [0.0, 0.1, 0.2],
    'n_hidden_layers': [1, 2],
    'hidden_units': [32, 64, 128],
    'activation_encoder': ['relu', 'leaky_relu'],
    'activation_decoder': ['relu', 'sigmoid']
    }

# Output the search space for verification
    print("\nHyperparameter Search Space:")
    for param, values in search_space_3.items():
        print(f"{param}: {values}")# 
    print("\n" + "="*50)    

    # Prepare validation split for Optuna objective
    x_fit, x_val = train_test_split(X_train, test_size=0.2, random_state=42, shuffle=True)

    # Define Optuna objective function
    def objective(trial):
        """Objective function for Optuna optimization."""
        # Sample hyperparameters
        latent_dim = trial.suggest_categorical('latent_dim', search_space_3['latent_dim'])
        learning_rate = trial.suggest_loguniform('learning_rate', search_space_3['learning_rate'][1], search_space_3['learning_rate'][0])
        batch_size = trial.suggest_categorical('batch_size', search_space_3['batch_size'])
        dropout_rate = trial.suggest_categorical('dropout_rate', search_space_3['dropout_rate'])
        n_hidden_layers = trial.suggest_categorical('n_hidden_layers', search_space_3['n_hidden_layers'])
        hidden_units = trial.suggest_categorical('hidden_units', search_space_3['hidden_units'])
        activation_encoder = trial.suggest_categorical('activation_encoder', search_space_3['activation_encoder'])
        activation_decoder = trial.suggest_categorical('activation_decoder', search_space_3['activation_decoder'])  

        # Build model with sampled hyperparameters
        model = build_autoencoder(
            input_dim=X_train.shape[1],
            latent_dim=latent_dim,
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            n_hidden_layers=n_hidden_layers,
            activation_encoder=activation_encoder,
            activation_decoder=activation_decoder
        )

        # Compile model
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='mse')

        # Train model with early stopping
        history = model.fit(
            x_fit, x_fit,
            validation_data=(x_val, x_val),
            epochs=30,
            batch_size=batch_size,
            callbacks=[
                keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
            ],
            verbose=0
        )

        # Return validation loss for minimization
        return min(history.history['val_loss'])

    # Create and run Optuna study
    print("Starting Optuna Bayesian optimization...")
    study3 = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study3.optimize(objective, n_trials=N_TRIALS, timeout=800)  # 50 trials or 15 minutes max

    # Display results
    print("\n" + "="*50)
    print("OPTUNA BAYESIAN OPTIMIZATION COMPLETE")
    print("="*50)
    print(f"Number of finished trials: {len(study3.trials)}")

    print("\nBest trial:")
    trial = study3.best_trial
    print(f"  Value (validation loss): {trial.value:.6f}")
    print("  Parameters:")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
    
    # Train the best model on the full training set with early stopping
    print("\nPreparing best model for full training...")
    best_params = study3.best_params
    best_model = build_autoencoder(
        input_dim=X_train.shape[1],
        latent_dim=best_params['latent_dim'],
        hidden_units=best_params['hidden_units'],
        dropout_rate=best_params['dropout_rate'],
        n_hidden_layers=best_params['n_hidden_layers'],
        activation_encoder=best_params['activation_encoder'],
        activation_decoder=best_params['activation_decoder']
    )

    best_model.compile(optimizer=keras.optimizers.Adam(learning_rate=best_params['learning_rate']), loss='mse')

    best_history = best_model.fit(
        X_train, X_train,
        validation_split=0.1,
        epochs=100,
        batch_size=best_params['batch_size'],
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0
    )
    
    # Best Model Summary
    print("\nBest Model Summary:")
    best_model.summary()
      
    # Save best model parameters to a text file
    print("\nSaving best model parameters to disk...")
    with open(models_dir / "best_model_params_v3.txt", "w") as f:
        f.write("Best Model Parameters:\n")
        for key, value in best_params.items():
            f.write(f"{key}: {value}\n")    
    print("Best model parameters saved successfully.")
    
    
    # Save the best model to disk
    print("\nSaving best model to disk...")
    best_model.save(models_dir / "autoencoder_best_model_v3.keras")
    print("Best model saved successfully.")
    
    # Save the final study for future reference
    print("\nSaving Optuna study to disk...")
    joblib.dump(study3, models_dir / "optuna_study_v3.pkl")
    print("Optuna study saved successfully.")
    
    # Visualize optimization history
    try:
        fig = optuna.visualization.plot_optimization_history(study3)
        fig.show()
    except Exception as e:
        print(f"Could not generate optimization history plot: {e}")

    try:
        fig = optuna.visualization.plot_param_importances(study3)
        fig.show()
    except Exception as e:
        print(f"Could not generate parameter importance plot: {e}")
        
else:
    print("Loading best model data from disk...")
    final_model = keras.models.load_model(models_dir / "autoencoder_best_model_v3.keras")

    # load the Optuna study
    study3 = joblib.load(models_dir / "optuna_study_v3.pkl")
    print("Optuna study loaded successfully.")

    # Load best model parameters from the text file
    best_params = {}
    with open(models_dir / "best_model_params_v3.txt", "r") as f:
        lines = f.readlines()[1:]  # Skip the first line
        for line in lines:
            key, value = line.strip().split(": ")
            best_params[key] = float(value) if '.' in value else int(value)
    
    # Print best model parameters
    print("\nBest Model Parameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
                
    print("\nBest Model Summary:")
    final_model.summary()

[I 2026-07-30 00:21:18,313] A new study created in memory with name: no-name-a41babec-ef1a-4258-b5f5-8382a047983a



Hyperparameter Search Space:
latent_dim: [16, 32, 64]
learning_rate: [0.001, 0.0005, 0.0001]
batch_size: [128, 256, 512]
dropout_rate: [0.0, 0.1, 0.2]
n_hidden_layers: [1, 2]
hidden_units: [32, 64, 128]
activation_encoder: ['relu', 'leaky_relu']
activation_decoder: ['relu', 'sigmoid']

Starting Optuna Bayesian optimization...


/var/folders/cp/98185ksd5djf61ly4251cnw40000gn/T/ipykernel_28343/211595103.py:39: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', search_space_3['learning_rate'][1], search_space_3['learning_rate'][0])
[I 2026-07-30 00:21:25,576] Trial 0 finished with value: 0.01738680712878704 and parameters: {'latent_dim': 32, 'learning_rate': 0.0007571539027086855, 'batch_size': 128, 'dropout_rate': 0.0, 'n_hidden_layers': 2, 'hidden_units': 32, 'activation_encoder': 'leaky_relu', 'activation_decoder': 'relu'}. Best is trial 0 with value: 0.01738680712878704.
[I 2026-07-30 00:21:34,255] Trial 1 finished with value: 0.028158806264400482 and parameters: {'latent_dim': 32, 'learning_rate': 0.0006122295769282673, 'batch_size': 512, 'dropout_rate': 0.2, 'n_hidden_layers': 2, 'hidden_units': 


OPTUNA BAYESIAN OPTIMIZATION COMPLETE
Number of finished trials: 50

Best trial:
  Value (validation loss): 0.004699
  Parameters:
    latent_dim: 32
    learning_rate: 0.0008844496350335233
    batch_size: 512
    dropout_rate: 0.0
    n_hidden_layers: 1
    hidden_units: 128
    activation_encoder: leaky_relu
    activation_decoder: relu

Preparing best model for full training...

Best Model Summary:


Model: "autoencoder"

...

 Total params: 116,420 (454.77 KB)

 Trainable params: 38,806 (151.59 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 77,614 (303.18 KB)


Saving best model parameters to disk...
Best model parameters saved successfully.

Saving best model to disk...
Best model saved successfully.

Saving Optuna study to disk...
Optuna study saved successfully.


## Optimization Results Summary

Comparison of the best results obtained from each optimization approach.

In [ ]:
if RUN_MODE:
    # Build comparison data
    comparison_data = []

    # Add baseline results if available
    if 'v1_df' in locals() and len(v1_df) > 0:
        best_v1 = v1_df.nsmallest(1, 'val_loss').iloc[0]
        comparison_data.append({
            'Approach': 'Baseline Grid Search (v1)',
            'Best Val Loss': best_v1['val_loss'],
            'Latent Dim': best_v1['latent_dim'],
            'Learning Rate': best_v1['learning_rate'],
            'Batch Size': best_v1['batch_size'],
            'Epochs': best_v1['epochs']
        })

    # Add Optuna results if available
    if 'study2' in locals() and len(study2.trials) > 0:
        best_trial = study2.best_trial
        comparison_data.append({
            'Approach': 'Bayesian Optimization (v2)',
            'Best Val Loss': best_trial.value,
            'Latent Dim': best_trial.params.get('latent_dim', 'N/A'),
            'Learning Rate': best_trial.params.get('learning_rate', 'N/A'),
            'Batch Size': best_trial.params.get('batch_size', 'N/A'),
            'Dropout Rate': best_trial.params.get('dropout_rate', 'N/A'),
            'Hidden Layers': best_trial.params.get('n_hidden_layers', 'N/A'),
            'Hidden Units': best_trial.params.get('hidden_units', 'N/A')
        })

    # Add extended search results if available
    if 'study3' in locals() and len(study3.trials) > 0:
        best_trial = study3.best_trial
        comparison_data.append({
            'Approach': 'Extended Search (v3)',
            'Best Val Loss': best_trial.value,
            'Latent Dim': best_trial.params.get('latent_dim', 'N/A'),
            'Learning Rate': best_trial.params.get('learning_rate', 'N/A'),
            'Batch Size': best_trial.params.get('batch_size', 'N/A'),
            'Dropout Rate': best_trial.params.get('dropout_rate', 'N/A'),
            'Hidden Layers': best_trial.params.get('n_hidden_layers', 'N/A'),
            'Hidden Units': best_trial.params.get('hidden_units', 'N/A'),
            'Encoder Activation': best_trial.params.get('activation_encoder', 'N/A'),
            'Decoder Activation': best_trial.params.get('activation_decoder', 'N/A')
        })

    # Display comparison
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        print("Optimization Approach Comparison:")
        print("="*80)
        print(comparison_df.to_string(index=False))

        # Find the best approach (minimum validation loss)
        best_row = comparison_df.loc[comparison_df['Best Val Loss'].idxmin()]
        best_approach = best_row['Approach']
        best_val_loss = best_row['Best Val Loss']

        print(f"\nBest approach: {best_approach} with validation loss: {best_val_loss:.6f}")

        # Load the corresponding model
        if best_approach == 'Baseline Grid Search (v1)':
            model_path = models_dir / "autoencoder_best_model_v1.keras"
        elif best_approach == 'Bayesian Optimization (v2)':
            model_path = models_dir / "autoencoder_best_model_v2.keras"
        elif best_approach == 'Extended Search (v3)':
            model_path = models_dir / "autoencoder_best_model_v3.keras"
        else:
            raise ValueError(f"Unknown approach: {best_approach}")

        print(f"Loading best model from: {model_path}")
        best_model = keras.models.load_model(model_path)

        # Evaluate on test set
        print("\nEvaluating best model on test set...")
        threshold, y_pred, metrics = generate_anomaly_metrics_and_threshold(
            best_model, X_train, X_test, y_test, percentile=95
        )
        print(f"Test Set F1-Score: {metrics['f1_score']:.4f}")
        print(f"Test Set Accuracy: {metrics['accuracy']:.4f}")

        # Save the best overall model for future use (when RUN_MODE=False)
        best_overall_path = models_dir / "best_overall_model.keras"
        best_model.save(best_overall_path)
        print(f"Best overall model saved to: {best_overall_path}")
    else:
        print("No optimization results available for comparison.")
else:
    try:
        best_overall_path = models_dir / "best_overall_model.keras"
        print(f"Loading best overall model from: {best_overall_path}")
        best_model = keras.models.load_model(best_overall_path)
        print("\nEvaluating best overall model on test set...")
        threshold, y_pred, metrics = generate_anomaly_metrics_and_threshold(
            best_model, X_train, X_test, y_test, percentile=95
        )
        print(f"Test Set F1-Score: {metrics['f1_score']:.4f}")
        print(f"Test Set Accuracy: {metrics['accuracy']:.4f}")
    except Exception as e:
        print(f"Could not load best overall model: {e}")
        print("Please run the notebook with RUN_MODE=True first to generate the best overall model.")

Optimization Approach Comparison:
                  Approach  Best Val Loss  Latent Dim  Learning Rate  Batch Size  Epochs  Dropout Rate  Hidden Layers  Hidden Units Encoder Activation Decoder Activation
 Baseline Grid Search (v1)       0.015118        32.0       0.001000       128.0    20.0           NaN            NaN           NaN                NaN                NaN
Bayesian Optimization (v2)       0.005644        64.0       0.000668       512.0     NaN           0.0            1.0          64.0                NaN                NaN
      Extended Search (v3)       0.004699        32.0       0.000884       512.0     NaN           0.0            1.0         128.0         leaky_relu               relu

Best approach: Extended Search (v3) with validation loss: 0.004699
Loading best model from: /Users/kallestewart/Github/SEP740-CourseProject-G9-P19-AnomalyDetection/models/autoencoder_best_model_v3.keras

Evaluating best model on test set...
Test Set F1-Score: 0.9851
Test Set Accuracy: